# README
This notebook was used to create the variant comparison. 

This code is not used in the S2CF+C approach, but could be used to explore the variant comparison some more. Make sure to update the data columns, file paths and file names when using.

For a reader that is not familiar with python code and would just like to execute and explore the S2CF+C approach or comparison module, I would suggest to look into the S2CF and comparison module, and the corresponding util files since they are more cleaned up and commented on.

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from comparison_analysis.utils.variant_analysis import extract_worksessions_with_time, normalize_activity

VERSION = 14
# minimum number of cases per variant
MIMINUM_NUMBER_CASES_THRESHOLD = 2
DT_FORMAT = "%d %b %Y %H:%M:%S,%f"

dfs = []

input_folder_path = f''

endtime_file_paths = [

]


for file_path in endtime_file_paths: 

  # -----------------------------
  # LOAD & PREPARE DATA
  # -----------------------------
  df = pd.read_csv(file_path)

  df = df[['Datetime', 'End Datetime', 'Study ID', 'Activity']].copy()

  # Convert timestamps
  df['Datetime'] = pd.to_datetime(df['Datetime'], format=DT_FORMAT, errors='coerce')
  df['End Datetime'] = pd.to_datetime(df['End Datetime'], format=DT_FORMAT, errors='coerce')

  # Sort properly
  df = df.sort_values(['Study ID', 'Datetime'])

  df.rename(columns={'Study ID': 'case_id', 'Activity': 'activity'}, inplace=True)

  case_traces = {}
  case_groups = dict(tuple(df.groupby('case_id')))

  for case_id, group in case_groups.items():
      trace = tuple(group['activity'].tolist())
      case_traces[case_id] = trace

  variant_dict = defaultdict(list)

  for case_id, trace in case_traces.items():
      variant_dict[trace].append(case_id)

  
  variant_dict = {
      k: v for k, v in variant_dict.items()
      if len(v) >= MIMINUM_NUMBER_CASES_THRESHOLD
  }

  sorted_variants = sorted(
      variant_dict.items(),
      key=lambda x: len(x[1]),
      reverse=True
  )

  case_durations = {}

  for case_id, group in case_groups.items():
      sessions = extract_worksessions_with_time(group)

      total_duration = pd.Timedelta(0)

      for session in sessions:
          start_time = session[0]['Datetime']
          end_time = session[-1]['End Datetime']

          if pd.notnull(start_time) and pd.notnull(end_time):
              total_duration += (end_time - start_time)

      # store duration in minutes
      case_durations[case_id] = total_duration.total_seconds() / 60

  # -----------------------------
  # BUILD RESULT TABLE
  # -----------------------------
  rows = []
  variant_id = 1

  for variant, cases in sorted_variants:
      num_cases = len(cases)

      # --- Duration stats ---
      durations = [case_durations[c] for c in cases if c in case_durations]

      avg_duration = np.mean(durations) if durations else 0
      median_duration = np.median(durations) if durations else 0

      # Extract worksessions from variant (structure only)
      def extract_sessions_from_trace(trace):
          sessions = []
          current = []
          for act in trace:
              if act == "1_Startup":
                  current = [act]
              elif current:
                  current.append(act)
                  if act in ["7_Shutdown", "0B_Abrupt end error"]:
                      sessions.append(tuple(current))
                      current = []
          return sessions

      sessions = extract_sessions_from_trace(variant)

      for i, session in enumerate(sessions, start=1):
          activity_names = [normalize_activity(a) for a in session]

          rows.append({
              "Variant ID": variant_id,
              "Included Cases": num_cases,
              "Avg Duration (min)": round(avg_duration, 2),
              "Median Duration (min)": round(median_duration, 2),
              "Worksession Number": i,
              "Sequence": ", ".join(activity_names),
          })

      variant_id += 1

  result_df = pd.DataFrame(rows)

  # Keep table nicely ordered
  result_df = result_df.sort_values(["Variant ID", "Worksession Number"])

  # -----------------------------
  # DISPLAY RESULT
  # -----------------------------
  print(result_df.to_latex(index=False))
